# Representation analysis — Quantum-Aware Representation Learning

This notebook explores the completed A/B development pilot: fold 0, 668 training
and 167 validation patients, 50 epochs, and seeds 1729/1730/1731. It reads only
aggregate CSV files available in the repository. It does not train models, refit
transformations, read patient-level outputs, or access the locked test.

A improved after the smoke-test budget; B varied across seeds, and its frozen
linear probe retained strong discrimination. These are single-partition
observations, not evidence of quantum advantage. See the
[principal report](../reports/phase2/pilot_fold0_50e_verified/report.md) for the
complete protocol, results, provenance, and scientific conclusions.


## Context and methods

The primary evaluation is epoch 50, not the validation minimum. Three seeds on
one partition are not independent patient cohorts. Sample SD (`ddof=1`) describes
initialization variability, not a confidence interval.

Six plots use shared functions in `src/phase2/analysis_views.py`. Saved CSV values
are displayed directly; loading them is not an independent validation of the
experiment. Deep reconciliation against private development artifacts remains
available separately through `load_analysis_artifacts`; see the
[execution guide](README_representation_analysis.md).


### 1. Paths

Set `QARL_ANALYSIS_DIR` to a project-relative or absolute directory containing
the six saved CSVs. `QARL_PROJECT_ROOT` is optional when running from the
repository root or `notebooks/`. No run archive or checkpoint is required.


In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("QARL_PROJECT_ROOT", ".." if Path.cwd().name == "notebooks" else ".")).expanduser().resolve()
if not (PROJECT_ROOT / "src/phase2").is_dir():
    raise FileNotFoundError("Set QARL_PROJECT_ROOT to the repository root.")
sys.path.insert(0, str(PROJECT_ROOT))
ANALYSIS_DIR = Path(os.environ.get("QARL_ANALYSIS_DIR", "reports/phase2/pilot_fold0_50e_verified")).expanduser()
if not ANALYSIS_DIR.is_absolute():
    ANALYSIS_DIR = PROJECT_ROOT / ANALYSIS_DIR
print(f"Saved analysis: {ANALYSIS_DIR}")


### 2. Load saved summaries

Missing inputs produce an error listing the required paths. No replacement
results are generated. These aggregate files contain no patient identifiers,
individual predictions, or individual latent vectors.


In [ ]:
from src.phase2.analysis_views import read_csv, generalization_table, plot_generalization, plot_latent_spectrum, plot_pilot_figures
import matplotlib.pyplot as plt

names = ["metrics_by_seed", "descriptive_summary", "paired_comparisons",
         "learning_curves", "latent_spectrum", "angular_diagnostics"]
missing = [ANALYSIS_DIR / f"{name}.csv" for name in names if not (ANALYSIS_DIR / f"{name}.csv").is_file()]
if missing:
    raise FileNotFoundError("Missing saved summary CSV files:\n" + "\n".join(map(str, missing)))
tables = {name: read_csv(ANALYSIS_DIR / f"{name}.csv") for name in names}
curves = dict(tuple(tables["learning_curves"].groupby(["experiment", "seed"], sort=False)))
rendered_figures = plot_pilot_figures(curves)

def show_figure(name):
    display(rendered_figures[name])
    plt.close(rendered_figures[name])


## Results

### 3. Comparisons by model and seed

ROC-AUC measures overall ordering; AP is reported separately for ER+ and ER−.
Balanced accuracy uses a fixed threshold of 0.5, and log-loss evaluates assigned
probabilities. Do not interchange these metrics when describing an improvement.

In [ ]:
metrics_by_seed = tables["metrics_by_seed"]
summary = tables["descriptive_summary"]
display(summary[["experiment", "roc_auc_mean", "roc_auc_std", "log_loss_mean", "log_loss_std"]].round(4))
display(metrics_by_seed[["experiment", "seed", "roc_auc", "ap_er_positive", "ap_er_negative", "balanced_accuracy", "log_loss"]].round(4))

**Reference observation:** B has higher mean ROC-AUC than A, with greater
variability; A_classical has higher mean ROC-AUC than B.
**Possible interpretation:** representation and optimization contribute to the
difference. **Limitation:** this does not identify a causal circuit contribution
or stability across folds. **C0 hypothesis:** controlled latent geometry might
change stability; first evaluate the frozen A/B protocol without designing
against validation results.

### 4. Convergence

Dashed lines show online training BCE; solid lines show validation BCE at the
end of each epoch. A common axis supports magnitude comparisons. Training loss
averages over changing parameters within an epoch; it is not an evaluation of
the final checkpoint.

In [ ]:
show_figure("convergence")

**Observation:** A continues to improve after the two-epoch smoke test;
B_classical reduces training loss almost to zero while validation loss worsens.
**Interpretation:** consistent with an insufficient smoke-test budget and later
overfitting in trainable networks. **Limitation:** neither the validation minimum
nor a favorable curve authorizes changing the primary checkpoint.
**C0 hypothesis:** evaluate representation regularization as a prespecified
mechanism, rather than using these curves for hyperparameter search.

### 5. Generalization: a descriptive gap

The difference `validation BCE − online training BCE` uses saved losses.
**It is not a pure estimate of the generalization gap**, because it compares
different parameter states. This limitation remains explicit in the calculation
and figure labels.

In [ ]:
generalization = generalization_table(tables["learning_curves"])
figure = plot_generalization(tables["learning_curves"])
display(figure)
plt.close(figure)
display(generalization[generalization.epoch == 50].round(4))

**Observation:** the gap grows particularly in B_classical and also appears in B.
**Interpretation:** a pattern consistent with specialization to training; it
cannot be attributed to a single layer. **C0 hypothesis:** ask whether a geometric
constraint reduces specialization while retaining a classical-head comparison.
No constraint or early stopping is applied here.

### 6. Gradients and circuit redundancy

Norms are minibatch averages; the parameter map aggregates epoch RMS gradients.
Evolution panels use independent y-axis scales. Four final RZ rotations commute
with the propagated observable and are structurally redundant.

In [ ]:
show_figure("gradient_evolution")
show_figure("parameter_gradients")

**Observation:** the encoder and relevant quantum parameters receive gradients;
final RZ gradients remain near numerical precision.
**Interpretation:** backpropagation is not completely disconnected; nominal
parameter counts do not equal effective capacity. **Limitation:** a nonzero
gradient demonstrates neither predictive usefulness nor favorable conditioning.
**C0 hypothesis:** check whether a geometric term changes gradient distributions,
separating its encoder and circuit effects.

### 7. Centered latent spectrum

Singular values are shown relative to the largest value, using a common y-axis.
Numerical rank counts values satisfying `s_i > s_max × 1e-6`. Normalization
facilitates contraction comparisons but hides absolute scale; absolute values
are retained in the table.

In [ ]:
spectrum = tables["latent_spectrum"]
figure = plot_latent_spectrum(spectrum, split="train", checkpoint="final")
display(figure)
plt.close(figure)
display(spectrum[(spectrum.checkpoint == "final") & (spectrum.split == "train")].round(6))

**Observation:** B finishes with numerical rank 3 in two seeds, although four
coordinates enter the VQC. **Interpretation:** a nearly constant direction may
act as a learned bias; this does not demonstrate an implementation error.
**Limitation:** rank depends on the threshold and does not directly measure class
information. **C0 hypothesis:** assess contraction and separability separately
when studying geometric regularization; do not retrospectively impose rank four.

### 8. Angular saturation

Angle = `pi*tanh(z/2)`; saturation is `|tanh(z/2)| > 0.95` or `> 0.99`.
Rates pool patient × coordinate values; the table retains coordinate-level
statistics. For B_classical these are hypothetical angles: its head operates
on z without angular scaling.

Exact angular histograms require private individual latent arrays and are
available through the separate saved-output analysis CLI, not this notebook.


In [ ]:
show_figure("angular_saturation")
angles = tables["angular_diagnostics"]
display(angles[(angles.checkpoint == "final") & (angles.coordinate == "pooled")][
    ["experiment", "seed", "split", "angular_saturation_95", "angular_saturation_99", "tanh_derivative_below_001"]].round(6))

**Observation:** B's final saturation depends on seed and coordinate; A remains
fixed. Evolution panels have independent y-axis scales: read the axes and exact
rates rather than comparing heights across panels.
**Interpretation:** a smaller angular derivative may attenuate local gradients;
this does not establish saturation as the cause of overfitting or latent contraction.
**C0 hypothesis:** record geometry, angles, and gradients jointly. Thresholds
are descriptive, not criteria for changing the scaling.

### 9. B versus B_frozen_probe

The saved paired table reports differences in metrics and Spearman correlations.
Original scores were joined by patient ID with label equality verified. This
notebook reads those aggregate results; it does not repeat patient pairing.
Spearman correlation does not measure calibration or equivalence. Exact score
scatterplots require the private development predictions and remain available
through the separate saved-output analysis CLI.


In [ ]:
paired = tables["paired_comparisons"]
display(paired[["seed", "n_validation", "B_minus_A_roc_auc", "probe_minus_B_roc_auc", "spearman_B_probe", "probe_minus_B_log_loss"]].round(4))

**Observation:** the probe exceeds B's ROC-AUC for all three seeds, with Spearman
correlations approximately 0.61–0.69; its log-loss is worse.
**Interpretation:** z supports a competitive linear head for discrimination,
without identical probabilities. **Limitation:** z was learned jointly with the
VQC, and the probe uses a different optimizer/budget; this is not an ablation
removing the circuit from training. **C0 hypothesis:** retain the probe as a
control to distinguish representation changes from head-specific effects.

### 10. Observed resources

Network time includes per-epoch validation and monitoring. Preprocessing is
shared across seeds/controls for each representation: do not sum its repeated
column. Probe timing excludes prior B training.

In [ ]:
display(metrics_by_seed[["experiment", "seed", "seconds", "preprocessing_seconds", "circuit_sample_evaluations"]].round(4))

## Conclusions and limitations

The saved pilot shows initialization variability, overfitting patterns, and
latent contraction. Establish stability across development partitions before
C0; neither novelty nor quantum advantage is demonstrated. No architecture,
regularizer, checkpoint selection, or experimental protocol is changed here.

The notebook reproduces six aggregate figures from versioned CSVs. It does not
independently establish their provenance or reproduce individual score scatterplots
and latent histograms. Deep verification and those two private-data panels use
the original development archive separately. No locked-test data is required.

The source notebook has empty outputs. Save executed copies and optional HTML
under ignored `runs/`; see the execution guide.
